In [3]:
import duckdb
import pandas as pd
import os
import json
from dotenv import load_dotenv


In [4]:
def extrair_features_moodle(pasta_dados="export/", curso_id="10464"):
    """
    Varre os logs brutos do Moodle e consolida métricas comportamentais 
    por aluno usando processamento analítico do DuckDB.
    """
    
    query_features = f"""
        SELECT 
            username,
            -- 1. Features de Volume de Engajamento
            COUNT(*) AS total_cliques,
            
            -- Converte o timestamp Unix para Data e conta os dias distintos estudados
            COUNT(DISTINCT CAST(to_timestamp(CAST(timecreated AS BIGINT)) AS DATE)) AS dias_ativos,
            
            -- 2. Features Comportamentais (Ativo vs Passivo)
            SUM(CASE WHEN crud IN ('c', 'u') THEN 1 ELSE 0 END) AS interacoes_ativas,
            SUM(CASE WHEN crud = 'r' THEN 1 ELSE 0 END) AS interacoes_passivas,
            
            -- 3. Features de Foco em Componentes
            SUM(CASE WHEN component = 'mod_forum' THEN 1 ELSE 0 END) AS cliques_forum,
            SUM(CASE WHEN component = 'mod_quiz' THEN 1 ELSE 0 END) AS cliques_quiz,
            SUM(CASE WHEN component IN ('mod_resource', 'mod_book', 'mod_page') THEN 1 ELSE 0 END) AS cliques_materiais
            
        FROM read_csv_auto('{pasta_dados}mdl_logstore_standard_log.csv', ALL_VARCHAR=TRUE)
        WHERE username NOT IN ('0', '-1', '', 'nan') 
          AND username IS NOT NULL
          AND courseid = '{curso_id}'
        GROUP BY username
        ORDER BY total_cliques DESC
    """
    
    # Executa a query e já devolve um DataFrame Pandas limpinho
    df_features = duckdb.query(query_features).df()
    return df_features

df = extrair_features_moodle()
print(df.head())

                  username  total_cliques  dias_ativos  interacoes_ativas  \
0  user6442803380426375169          31212           86             5250.0   
1  user8540069828419911681          24996           71            22080.0   
2  user7619359643386511361          17247           39             4721.0   
3  user7959886181284446209           7180           38              212.0   
4  user2959087468848087041           6685           36              679.0   

   interacoes_passivas  cliques_forum  cliques_quiz  cliques_materiais  
0              25806.0        18262.0         460.0             1376.0  
1               2892.0         1240.0          50.0              260.0  
2              12519.0        15427.0          33.0              182.0  
3               6965.0         5871.0         139.0              228.0  
4               6002.0         4658.0          37.0               77.0  


In [5]:
def extrair_notas_medias(pasta_dados="export/"):
    """
    Extrai a nota mais recente de cada item para cada aluno
    e calcula a nota média final (Target).
    """
    query_notas = f"""
        WITH NotasRecentes AS (
            -- Passo 1: Pegar apenas a nota mais recente (última modificação) de cada atividade por aluno
            SELECT 
                username,
                itemid,
                arg_max(CAST(finalgrade AS FLOAT), CAST(timemodified AS BIGINT)) as nota_recente
            FROM read_csv_auto('{pasta_dados}mdl_grade_grades_history.csv', ALL_VARCHAR=TRUE)
            WHERE finalgrade NOT IN ('', 'nan', 'None') AND finalgrade IS NOT NULL
              AND username NOT IN ('0', '-1', '', 'nan')
            GROUP BY username, itemid
        )
        -- Passo 2: Calcular a nota média por aluno a partir de suas notas mais recentes
        SELECT 
            username,
            AVG(nota_recente) AS nota_media
        FROM NotasRecentes
        GROUP BY username
        ORDER BY nota_media DESC
    """
    
    # Executa a query e retorna o DataFrame
    return duckdb.query(query_notas).df()

# Executa a função e guarda o DataFrame com o nosso "Target"
df_target = extrair_notas_medias()

# Exibe as primeiras linhas para validação
display(df_target.head())

,username,nota_media
0,user3196961191700201473,80.0
1,user7232449863873462273,80.0
2,user2111797327378251777,80.0
3,user7863879879668793345,80.0
4,user8749729812320354305,80.0


In [ ]:
from functools import reduce

# 1. Configuração

load_dotenv()

export_path = os.getenv("PASTA_DADOS")

atividades_path = os.path.join(export_path, "atividades.json")
nivel_desordem_path = os.path.join(export_path, "nivel_desordem.json")
visualizacoes_por_objeto_path = os.path.join(export_path, "visualizacoes_por_objeto.json")
proporcao_visualizacoes_path = os.path.join(export_path, "proporcao_visualizacoes_por_atividade.json")
pontuacao_path = os.path.join(export_path, "pontuacao.json")
porcentagem_curso_path = os.path.join(export_path, "porcentagem_curso_acessada.json")
tentativas_por_questionario_path = os.path.join(export_path, "tentativas_por_questionario.json")
tempo_resposta_path = os.path.join(export_path, "tempo_resposta.json")
tempo_total_gasto_path = os.path.join(export_path, "tempo_total_gasto_em_visitas_reais.json")

# 2.1. Desordem por Sessão

with open(nivel_desordem_path, 'r', encoding='utf-8') as f:
    desordem_json = json.load(f)
dados_desordem = {}
for id_sessao, dados_sessao in desordem_json.get("Course 10464", {}).get("sessoes", {}).items():
    for username, valor in dados_sessao.get("nivel_desordem_por_usuario", {}).items():
        if username not in dados_desordem:
            dados_desordem[username] = {"username": username}
        dados_desordem[username][f"desordem_sessao_{id_sessao}"] = valor
df_desordem = pd.DataFrame(list(dados_desordem.values()))

# 2.2. Pontuação
with open(pontuacao_path, 'r', encoding='utf-8') as f:
    pont_data = json.load(f)
df_pontuacao = pd.DataFrame(pont_data)[['usuario', 'pontuacao']]
df_pontuacao = df_pontuacao.rename(columns={'usuario': 'username'}) # Padronizando a chave

# 2.3. Porcentagem do Curso Acessada (Desdobrando geral e sessões)
with open(porcentagem_curso_path, 'r', encoding='utf-8') as f:
    perc_json = json.load(f)
dados_perc = []
for username, dados_aluno in perc_json.get('Course 10464', {}).items():
    registro = {'username': username, 'porcentagem_geral': dados_aluno.get('geral', 0.0)}
    # Expande as sessões também
    for id_sessao, valor_sessao in dados_aluno.get('sessoes', {}).items():
        registro[f'porcentagem_sessao_{id_sessao}'] = valor_sessao
    dados_perc.append(registro)
df_porcentagem = pd.DataFrame(dados_perc)

# 2.4. Tentativas por Questionário (Somando tudo por aluno)
with open(tentativas_por_questionario_path, 'r', encoding='utf-8') as f:
    tentativas_data = json.load(f)
df_tentativas = pd.DataFrame(tentativas_data)
df_tentativas = df_tentativas.groupby('usuario')['total_tentativas'].sum().reset_index()
df_tentativas = df_tentativas.rename(columns={'usuario': 'username', 'total_tentativas': 'soma_tentativas_quizzes'})

# 2.5. Tempo de Resposta (Transformando PT...S em segundos)
with open(tempo_resposta_path, 'r', encoding='utf-8') as f:
    tempo_resp_data = json.load(f)
df_tempo_resp = pd.DataFrame(tempo_resp_data)

# Solução à prova de falhas usando Regex para extrair (Dias, Horas, Minutos, Segundos)
regex_tempo = r'P(?:(\d+)D)?T(?:(\d+)H)?(?:(\d+)M)?(?:(\d+)S)?'
extraido_resp = df_tempo_resp['tempo_resposta'].str.extract(regex_tempo).fillna(0).astype(int)

# D*86400 + H*3600 + M*60 + S
df_tempo_resp['tempo_resp_segundos'] = (
    extraido_resp[0] * 86400 + 
    extraido_resp[1] * 3600 + 
    extraido_resp[2] * 60 + 
    extraido_resp[3]
)

# Tirando a média de tempo de resposta por aluno
df_tempo_resp = df_tempo_resp.groupby('usuario')['tempo_resp_segundos'].mean().reset_index()
df_tempo_resp = df_tempo_resp.rename(columns={'usuario': 'username', 'tempo_resp_segundos': 'media_tempo_resposta_seg'})


# ==========================================
# 2.6. Tempo Total Gasto na Plataforma
# ==========================================
with open(tempo_total_gasto_path, 'r', encoding='utf-8') as f:
    tempo_gasto_data = json.load(f)
df_tempo_gasto = pd.DataFrame(tempo_gasto_data)

# Aplicando a mesma extração Regex (O Moodle pode gerar números enormes como PT436985S aqui)
extraido_gasto = df_tempo_gasto['tempo_passado'].str.extract(regex_tempo).fillna(0).astype(int)

# Calculando Segundos
df_tempo_gasto['tempo_passado_seg'] = (
    extraido_gasto[0] * 86400 + 
    extraido_gasto[1] * 3600 + 
    extraido_gasto[2] * 60 + 
    extraido_gasto[3]
)

# Somando todo o tempo das visitas do aluno
df_tempo_gasto = df_tempo_gasto.groupby('usuario')['tempo_passado_seg'].sum().reset_index()
df_tempo_gasto = df_tempo_gasto.rename(columns={'usuario': 'username', 'tempo_passado_seg': 'tempo_total_ambiente_seg'})


# ==========================================
# 3. O GRANDE MERGE (Unindo tudo pelo Username)
# ==========================================
# Colocamos todos os DataFrames em uma lista
dataframes_para_unir = [
    df_desordem, 
    df_pontuacao, 
    df_porcentagem, 
    df_tentativas, 
    df_tempo_resp, 
    df_tempo_gasto,
    df
]

# Usamos reduce para fazer o OUTER JOIN em todos eles de forma sequencial
df_final = reduce(lambda left, right: pd.merge(left, right, on='username', how='outer'), dataframes_para_unir)

# ==========================================
# 4. TRATAMENTO FINAL DE NULOS
# ==========================================

df_final = df_final.fillna(0.0)

# Para garantir que não haja usernames nulos ou vazios
df_final = df_final[df_final['username'] != 0.0]

print(f"✅ Matriz de features construída com sucesso! Total de alunos: {len(df_final)}")
print(f"📊 Total de features (colunas) geradas: {len(df_final.columns)}")

# Exibe a super tabela
display(df_final.head())

df_final.to_csv(os.path.join(export_path, "matriz_comportamental.csv"), index=False)

username                       0
desordem_sessao_58964        134
desordem_sessao_58966        941
desordem_sessao_58965        160
desordem_sessao_58968       1031
desordem_sessao_58967       1177
pontuacao                    980
porcentagem_geral             27
porcentagem_sessao_58964      27
porcentagem_sessao_58966      27
porcentagem_sessao_58965      27
porcentagem_sessao_58968      27
porcentagem_sessao_58969      27
porcentagem_sessao_58967      27
soma_tentativas_quizzes      994
media_tempo_resposta_seg     824
tempo_total_ambiente_seg       0
total_cliques                  0
dias_ativos                    0
interacoes_ativas              0
interacoes_passivas            0
cliques_forum                  0
cliques_quiz                   0
cliques_materiais              0
dtype: int64
✅ Matriz de features construída com sucesso! Total de alunos: 2168
📊 Total de features (colunas) geradas: 24


,username,desordem_sessao_58964,desordem_sessao_58966,desordem_sessao_58965,desordem_sessao_58968,desordem_sessao_58967,pontuacao,porcentagem_geral,porcentagem_sessao_58964,porcentagem_sessao_58966,...,soma_tentativas_quizzes,media_tempo_resposta_seg,tempo_total_ambiente_seg,total_cliques,dias_ativos,interacoes_ativas,interacoes_passivas,cliques_forum,cliques_quiz,cliques_materiais
0,user1002410981378228225,0.0000,0.0,0.0000,0.000,0.0000,0.00,5.56,0.0,18.18,...,0.0,0.000000,1797805,28,1,7.0,21.0,0.0,0.0,2.0
1,user1003176640903118849,0.0000,0.0,0.3138,0.000,0.0000,8.67,14.81,0.0,0.00,...,1.0,585.500000,4674426,75,4,20.0,55.0,0.0,7.0,24.0
2,user1011513786604978177,0.3869,0.0,0.7127,0.000,0.0000,9.50,38.89,100.0,54.55,...,4.0,1921.285714,1121166,211,2,59.0,152.0,18.0,26.0,61.0
3,user101338288765272065,0.0000,0.0,0.3869,0.000,0.0000,0.00,12.96,75.0,0.00,...,0.0,0.000000,940302,51,1,10.0,41.0,19.0,0.0,10.0
4,user1022941294420295681,0.3869,0.0,0.4106,0.865,0.5803,9.74,66.67,100.0,63.64,...,6.0,484.500000,831258,395,6,110.0,279.0,46.0,41.0,45.0
